# Entrenamiento YOLO11 para clasificar huevos

Este cuaderno prepara un dataset YOLO con las clases detectadas en las etiquetas, separa un conjunto final aislado, entrena varias configuraciones y conserva los tres mejores modelos.

Estructura esperada en Google Drive o en el proyecto local:

- `models/Data/images/`
- `models/Data/labels/`

Las etiquetas usan IDs YOLO consecutivos desde `0`. El cuaderno detecta automáticamente cuántas clases existen en el dataset y genera los YAML con ese número.

In [1]:
# Si trabajas en Colab, ejecuta esta línea una sola vez.
# %pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pandas matplotlib seaborn

import json
import random
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image
from ultralytics import YOLO
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PROJECT_CANDIDATES = [
    Path('/content/Huevos'),
    Path('/home/lukga/Programacion/CienciaDeDatos/clase/Huevos'),
    Path.cwd() / 'Huevos',
]
PROJECT_ROOT = next(
    (candidate for candidate in PROJECT_CANDIDATES if (candidate / 'models').is_dir()),
    PROJECT_CANDIDATES[0],
)
DATA_ROOT = PROJECT_ROOT / 'models' / 'Data'
DATASET_ROOT = PROJECT_ROOT / 'models' / 'yolo_dataset_final_holdout'
FINAL_TEST_ROOT = PROJECT_ROOT / 'final_test_holdout'
RUNS_ROOT = PROJECT_ROOT / 'runs_colab'
OUTPUT_ROOT = PROJECT_ROOT / 'top_models'

WEIGHTS = 'yolo11n.pt'
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
SEED = 42
TRIALS = 15
EPOCHS = 500
IMAGE_SIZE = 640
BATCH_SIZE = 32 if torch.cuda.is_available() else 4
WORKERS = 2
PATIENCE = 20
FINAL_TEST_RATIO = 0.20
INNER_VAL_RATIO = 0.20
REBUILD_DATASET = True
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TARGET_CLASS_IDS = {0, 1}

# El dataset actual contiene las clases 0 y 1.
CLASS_NAMES = ['clase_0', 'clase_1']

if not (DATA_ROOT / 'images').is_dir() or not (DATA_ROOT / 'labels').is_dir():
    raise FileNotFoundError(f'Se esperan las carpetas {DATA_ROOT / "images"} y {DATA_ROOT / "labels"}.')

for directory in (DATASET_ROOT, FINAL_TEST_ROOT, RUNS_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print('Proyecto:', PROJECT_ROOT)
print('Datos:', DATA_ROOT)
print('Dispositivo:', DEVICE)
print('Clases:', dict(enumerate(CLASS_NAMES)))

Python: 3.13.0 (main, Sep 11 2026, 18:34:43) [GCC 15.2.0]
Torch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Proyecto: /home/lukga/Programacion/CienciaDeDatos/clase/Huevos
Datos: /home/lukga/Programacion/CienciaDeDatos/clase/Huevos/models/Data
Dispositivo: 0
Clases: {0: 'clase_0', 1: 'clase_1'}


In [2]:
def read_yolo_label(label_path: Path):
    normalized = []
    observed_classes = set()

    for line_number, line in enumerate(label_path.read_text().splitlines(), 1):
        values = line.split()
        if not values:
            continue
        if len(values) < 5:
            raise ValueError(f'{label_path.name}:{line_number}: se esperan al menos 5 valores')

        try:
            class_id = int(values[0])
            coordinates = [float(value) for value in values[1:]]
        except ValueError as error:
            raise ValueError(f'{label_path.name}:{line_number}: valor no numerico') from error

        if class_id not in TARGET_CLASS_IDS:
            raise ValueError(
                f'{label_path.name}:{line_number}: clase {class_id} fuera de {TARGET_CLASS_IDS}'
            )
        if len(coordinates) != 4 and (len(coordinates) < 6 or len(coordinates) % 2 != 0):
            raise ValueError(f'{label_path.name}:{line_number}: formato YOLO invalido')
        if any(value < 0 or value > 1 for value in coordinates):
            raise ValueError(f'{label_path.name}:{line_number}: coordenadas fuera de [0, 1]')

        if len(coordinates) == 4:
            center_x, center_y, width, height = coordinates
        else:
            x_values = coordinates[::2]
            y_values = coordinates[1::2]
            x_min, x_max = min(x_values), max(x_values)
            y_min, y_max = min(y_values), max(y_values)
            center_x = (x_min + x_max) / 2
            center_y = (y_min + y_max) / 2
            width = x_max - x_min
            height = y_max - y_min

        if width <= 0 or height <= 0:
            raise ValueError(f'{label_path.name}:{line_number}: bounding box vacia')

        normalized.append(
            f'{class_id} {center_x:.8f} {center_y:.8f} {width:.8f} {height:.8f}'
        )
        observed_classes.add(class_id)

    return normalized, observed_classes



def collect_pairs(data_root: Path):
    global TARGET_CLASS_IDS, CLASS_NAMES

    image_root = data_root / 'images'
    label_root = data_root / 'labels'
    label_paths = {path.stem: path for path in label_root.glob('*.txt')}
    pairs = []
    skipped = []
    class_counts = Counter()

    detected_class_ids = set()
    for label_path in label_paths.values():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if values:
                detected_class_ids.add(int(values[0]))

    if not detected_class_ids:
        raise RuntimeError('No se encontraron clases en los archivos de etiquetas.')
    if detected_class_ids != set(range(max(detected_class_ids) + 1)):
        raise RuntimeError(
            f'Las clases deben usar IDs consecutivos desde 0; se encontraron {sorted(detected_class_ids)}'
        )

    TARGET_CLASS_IDS = detected_class_ids
    if len(CLASS_NAMES) < len(TARGET_CLASS_IDS):
        CLASS_NAMES.extend(
            f'clase_{index}'
            for index in range(len(CLASS_NAMES), len(TARGET_CLASS_IDS))
        )
    if len(CLASS_NAMES) > len(TARGET_CLASS_IDS):
        print(
            f'Advertencia: el dataset contiene {len(TARGET_CLASS_IDS)} clases; '
            f'se ignoraran nombres sobrantes: {CLASS_NAMES[len(TARGET_CLASS_IDS):]}'
        )
        CLASS_NAMES = CLASS_NAMES[:len(TARGET_CLASS_IDS)]

    for image_path in sorted(image_root.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label_path = label_paths.get(image_path.stem)
        if label_path is None:
            skipped.append(f'Falta etiqueta: {image_path.name}')
            continue
        try:
            normalized, observed_classes = read_yolo_label(label_path)
            with Image.open(image_path) as image:
                image.verify()
            pairs.append((image_path, normalized))
            class_counts.update(observed_classes)
        except (OSError, ValueError) as error:
            skipped.append(f'{image_path.name}: {error}')

    if not pairs:
        raise RuntimeError('No se encontraron pares imagen/etiqueta validos.')

    print(f'Pares validos: {len(pairs)}')
    print(f'Archivos omitidos: {len(skipped)}')
    print('Objetos por clase:', {CLASS_NAMES[index]: class_counts[index] for index in sorted(class_counts)})
    return pairs, skipped

pairs, skipped = collect_pairs(DATA_ROOT)

Pares validos: 22945
Archivos omitidos: 0
Objetos por clase: {'clase_0': 9119, 'clase_1': 13727}


In [3]:
SPLIT_MANIFEST = FINAL_TEST_ROOT / 'split_manifest.json'

if REBUILD_DATASET or not SPLIT_MANIFEST.exists():
    shuffled = pairs.copy()
    random.Random(SEED).shuffle(shuffled)
    final_test_count = max(1, round(len(shuffled) * FINAL_TEST_RATIO))
    final_test_pairs = shuffled[:final_test_count]
    training_pairs = shuffled[final_test_count:]
    SPLIT_MANIFEST.write_text(json.dumps({
        'seed': SEED,
        'final_test_ratio': FINAL_TEST_RATIO,
        'source_count': len(pairs),
        'final_test_images': sorted(path.name for path, _ in final_test_pairs),
    }, indent=2))
else:
    manifest = json.loads(SPLIT_MANIFEST.read_text())
    final_test_names = set(manifest['final_test_images'])
    final_test_pairs = [(path, labels) for path, labels in pairs if path.name in final_test_names]
    training_pairs = [(path, labels) for path, labels in pairs if path.name not in final_test_names]

if len(training_pairs) < 2:
    raise RuntimeError('Se necesitan al menos dos imagenes fuera del conjunto final.')

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
(DATASET_ROOT / 'images' / 'train').mkdir(parents=True)
(DATASET_ROOT / 'images' / 'val').mkdir(parents=True)
(DATASET_ROOT / 'labels' / 'train').mkdir(parents=True)
(DATASET_ROOT / 'labels' / 'val').mkdir(parents=True)

training_shuffled = training_pairs.copy()
random.Random(SEED).shuffle(training_shuffled)
val_count = max(1, round(len(training_shuffled) * INNER_VAL_RATIO))
split_pairs = {'val': training_shuffled[:val_count], 'train': training_shuffled[val_count:]}
if not split_pairs['train']:
    raise RuntimeError('La particion de entrenamiento quedo vacia.')

for split, split_items in split_pairs.items():
    for image_path, labels in split_items:
        shutil.copy2(image_path, DATASET_ROOT / 'images' / split / image_path.name)
        (DATASET_ROOT / 'labels' / split / f'{image_path.stem}.txt').write_text(
            '\n'.join(labels) + ('\n' if labels else '')
        )

yaml_lines = [
    f'path: {DATASET_ROOT.resolve().as_posix()}',
    'train: images/train',
    'val: images/val',
    'test: images/val',
    f'nc: {len(CLASS_NAMES)}',
    'names:',
    *[f'  {index}: {name}' for index, name in enumerate(CLASS_NAMES)],
]
(DATASET_ROOT / 'dataset.yaml').write_text('\n'.join(yaml_lines) + '\n')

if FINAL_TEST_ROOT.exists():
    shutil.rmtree(FINAL_TEST_ROOT)
(FINAL_TEST_ROOT / 'images').mkdir(parents=True)
(FINAL_TEST_ROOT / 'labels').mkdir(parents=True)
for image_path, labels in final_test_pairs:
    shutil.copy2(image_path, FINAL_TEST_ROOT / 'images' / image_path.name)
    (FINAL_TEST_ROOT / 'labels' / f'{image_path.stem}.txt').write_text(
        '\n'.join(labels) + ('\n' if labels else '')
    )
final_yaml = [
    f'path: {FINAL_TEST_ROOT.resolve().as_posix()}',
    'train: images',
    'val: images',
    'test: images',
    f'nc: {len(CLASS_NAMES)}',
    'names:',
    *[f'  {index}: {name}' for index, name in enumerate(CLASS_NAMES)],
]
(FINAL_TEST_ROOT / 'dataset.yaml').write_text('\n'.join(final_yaml) + '\n')

print(f'Entrenamiento + validacion: {len(training_pairs)}')
print(f'Evaluacion final aislada: {len(final_test_pairs)}')
print('Dataset YAML:', DATASET_ROOT / 'dataset.yaml')

Entrenamiento + validacion: 18356
Evaluacion final aislada: 4589
Dataset YAML: /home/lukga/Programacion/CienciaDeDatos/clase/Huevos/models/yolo_dataset_final_holdout/dataset.yaml


## Busqueda aleatoria y entrenamiento reanudable

Cada prueba guarda `last.pt`, `best.pt` y sus métricas. Si Colab se desconecta, vuelve a ejecutar la celda: una prueba con `last.pt` continuará desde el último checkpoint.

In [4]:
SEARCH_SPACE = {
    'lr0': [0.0005, 0.001, 0.005],
    'lrf': [0.01, 0.05, 0.1],
    'weight_decay': [0.0001, 0.0005, 0.001],
    'hsv_h': [0.01, 0.02],
    'hsv_s': [0.5, 0.7],
    'hsv_v': [0.3, 0.5],
    'degrees': [0.0, 5.0],
    'scale': [0.3, 0.5],
    'fliplr': [0.0, 0.5],
    'mosaic': [0.5, 1.0],
    'mixup': [0.0, 0.1],
    'dropout': [0.0, 0.1],
}

RESULTS_FILE = OUTPUT_ROOT / 'search_results.json'
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
completed_trials = {item['trial'] for item in results}

for trial in range(1, TRIALS + 1):
    if trial in completed_trials:
        print(f'Trial {trial}/{TRIALS} ya registrado; se omite.')
        continue

    run_name = f'random_trial_{trial:03d}'
    run_dir = RUNS_ROOT / run_name
    last_checkpoint = run_dir / 'weights' / 'last.pt'
    best_checkpoint = run_dir / 'weights' / 'best.pt'
    previous_best = sorted(results, key=lambda item: item['score'], reverse=True)[:3]
    initial_weights = WEIGHTS if not previous_best else previous_best[(trial - 1) % len(previous_best)]['weights']
    config = {
        name: random.Random(SEED + trial).choice(values)
        for name, values in SEARCH_SPACE.items()
    }

    if last_checkpoint.exists():
        print(f'Reanudando {run_name} desde {last_checkpoint}')
        model = YOLO(str(last_checkpoint))
        model.train(resume=True)
        initialization = str(last_checkpoint)
    else:
        print(f'Iniciando {run_name} desde {initial_weights}')
        model = YOLO(initial_weights)
        model.train(
            data=str(DATASET_ROOT / 'dataset.yaml'),
            epochs=EPOCHS,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            workers=WORKERS,
            patience=PATIENCE,
            seed=SEED + trial,
            project=str(RUNS_ROOT),
            name=run_name,
            exist_ok=True,
            optimizer='AdamW',
            amp=True,
            save_period=5,
            **config,
        )
        initialization = initial_weights

    if not best_checkpoint.exists():
        raise FileNotFoundError(f'No se encontro {best_checkpoint}')
    metrics = YOLO(str(best_checkpoint)).val(
        data=str(DATASET_ROOT / 'dataset.yaml'),
        split='val',
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )
    score = float(metrics.box.map)
    results.append({
        'trial': trial,
        'score': score,
        'weights': str(best_checkpoint),
        'initialization': initialization,
        'hyperparameters': config,
    })
    results = sorted(results, key=lambda item: item['score'], reverse=True)
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f'Trial {trial}/{TRIALS}: mAP50-95 de validacion={score:.5f}')

    # Descomenta si quieres separar las pruebas diez minutos.
    # if trial < TRIALS:
    #     time.sleep(600)

results = sorted(results, key=lambda item: item['score'], reverse=True)
pd.DataFrame([
    {'trial': item['trial'], 'mAP50-95 validacion': item['score']}
    for item in results
])

Iniciando random_trial_001 desde yolo11n.pt
New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.148 🚀 Python-3.13.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/lukga/Programacion/CienciaDeDatos/clase/Huevos/models/yolo_dataset_final_holdout/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz

KeyboardInterrupt: 

In [ ]:
# Guarda los tres mejores modelos y copia el ganador a models/best.pt.
for rank, item in enumerate(results[:3], start=1):
    destination = OUTPUT_ROOT / f'model_{rank}'
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(item['weights'], destination / 'best.pt')
    metadata = {
        'rank': rank,
        'trial': item['trial'],
        'metric': 'mAP50-95',
        'score': item['score'],
        'hyperparameters': item['hyperparameters'],
        'class_names': CLASS_NAMES,
    }
    (destination / 'configuration.json').write_text(json.dumps(metadata, indent=2))

if results:
    shutil.copy2(results[0]['weights'], PROJECT_ROOT / 'models' / 'best.pt')
    print('Ganador:', PROJECT_ROOT / 'models' / 'best.pt')
print('Modelos guardados en:', OUTPUT_ROOT)

## Graficas y evaluacion final

La métrica del conjunto final se calcula solo después de elegir el modelo con la validación interna.

In [ ]:
if not results:
    raise RuntimeError('Ejecuta primero la celda de entrenamiento.')

best_trial = results[0]
best_run = RUNS_ROOT / f"random_trial_{best_trial['trial']:03d}"
plot_files = [
    'results.png',
    'confusion_matrix.png',
    'PR_curve.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, filename in zip(axes.flat, plot_files):
    path = best_run / filename
    if path.exists():
        axis.imshow(Image.open(path))
        axis.set_title(filename)
    else:
        axis.text(0.5, 0.5, f'No generado: {filename}', ha='center', va='center')
    axis.axis('off')
plt.tight_layout()
plt.show()

history_path = best_run / 'results.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()
    display(history.tail())
    loss_columns = [column for column in history.columns if 'loss' in column]
    if loss_columns:
        history.plot(x='epoch', y=loss_columns, figsize=(12, 5), title='Perdidas por epoca')
        plt.grid()
        plt.show()

best_model = YOLO(results[0]['weights'])
final_metrics = best_model.val(
    data=str(FINAL_TEST_ROOT / 'dataset.yaml'),
    split='test',
    imgsz=IMAGE_SIZE,
    device=DEVICE,
)
print(f"Modelo ganador: trial {results[0]['trial']}")
print(f"mAP50-95 final: {float(final_metrics.box.map):.5f}")
print(f"mAP50 final: {float(final_metrics.box.map50):.5f}")

final_images = sorted((FINAL_TEST_ROOT / 'images').glob('*'))
sample_images = final_images[:min(6, len(final_images))]
if sample_images:
    predictions = best_model.predict(
        source=[str(path) for path in sample_images],
        conf=0.35,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )
    rows = (len(predictions) + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(18, 6 * rows))
    axes = [axes] if rows == 1 else axes
    for axis in axes.flat:
        axis.axis('off')
    for axis, prediction in zip(axes.flat, predictions):
        axis.imshow(prediction.plot()[..., ::-1])
    plt.tight_layout()
    plt.show()